# ♟️ Chess Numba — Classical HCE Texel / SPSA 分區調參工作台

在完成 `evaluation.py` 的 **8 項錯誤修復與順序修正**（`BUG-1..8`、`ORD-1..3`、`KP-DISCONT`）後，本 Notebook 提供**單一 Kernel、免重載資料、免重編譯 Numba** 的互動式區塊調參流程。

### 🛡️ 核心設計：自動尺度對齊 ($K^*$)、模式切換、SF19 混合標籤與記憶體安全
1. **`dataset_epd_wdl_20m.npz`（~19,630,000 局面）無洩漏三切分**：
   - **`~16.6M` Train Pool**（16 GB RAM 自動抽樣 `10M` 防 Swap）：僅供 SPSA mini-batch 梯度估計。
   - **`1,000,000` Dev Split (`rng=0`)**：供 SPSA 過程報告、early stopping 與優化端 $K^*$ 校準（`dev_K*`）；而 `2M unseen holdout`、`val_pgn`、`val_zurichess` 則各自於其純 WDL baseline 分數上獨立擬合自身 $K^*$，徹底分離混合標籤與純 WDL 尺度（並已完全移除舊版 `K=1.000` 驗證）。
   - **`2,000,000` Unseen Holdout (`rng=42`)**：**完全不進訓練、也不進 early stop，且永遠維持純 WDL 評估**，作為第一道靜態驗收閘門。
2. **可切換獨立驗證集（`USE_PGN_VAL` / `USE_ZURICHESS_VAL`）**：
   - **`USE_PGN_VAL = True`（預設雙重驗證）**：同時以 `2M unseen holdout` + `val_pgn.npz`（1.59M 實戰局面，含 `castling!=0` 易位權桶與校準 $K^*$）把關。
   - **`USE_PGN_VAL = False`（純 Holdout 對照模式）**：略過 `val_pgn.npz` 載入與評估，僅依 `2M unseen holdout (@ K*)` 判定。
   - **`USE_ZURICHESS_VAL`（第三仲裁／Tie-Breaker 驗證集）**：設為 `True` 時載入 `val_zurichess.npz`（1.08M 局面，含 13.53% 易位權），與 `val_pgn` 互相仲裁易位相關與微幅誤差；當同目錄存在 `val_pgn.sf19.npz`、`val_zurichess.sf19.npz` 或覆蓋 Holdout 的 `dataset_epd_wdl_20m.sf19.npz` 時，**無論 `USE_SF_LABELS` 是否開啟，皆會於 100% 純 WDL 上額外輸出 `sf19-quiet (WDL)` 與 `sf19-tactical (WDL)` 子診斷**（絕不混改驗證目標）。
3. **可切換 SF19 搜尋評估＋WDL 線性混合標籤（`USE_SF_LABELS` & `SF_LAMBDA`）**：
   - 先在終端機執行 `python -m tuner.data_pipeline.annotate_sf_labels` 產生輕量側車檔 `dataset_epd_wdl_20m.sf19.npz`（~35 MB，支援 `Ctrl+C` 中斷續傳與 `--limit` 子集先行測試）。
   - 開啟 `USE_SF_LABELS = True` 後，Session 會在 `train_idx` 上自動擬合無偏 $K_{	ext{sf}}$，於記憶體動態合成 $y_{	ext{mix}} = (1 - \lambda)\, y_{	ext{wdl}} + \lambda\, \sigma(K_{	ext{sf}} \cdot 	ext{sf\_cp})$（並自動剔除 $|	ext{sf\_cp} - 	ext{sf\_d1\_cp}| > 180$ 的戰術跳分局面），同時保持 `2M unseen holdout` 與 `val_pgn` 為**純 WDL** 不變尺規。
4. **目標引擎選擇 (`TARGET_ENGINE`) 與零重載切換 (`session.reconfigure`)**：
   - `"classical"` 寫入 `chess_engine/classical/constants.py`；`"classical_old"` 寫入 `chess_engine/classical_old/constants.py`，輸出與備份自動隔離至 `tuner/runs/<engine>_<mode>[_sfXX]/`。
   - 修改 `# 0` 開關後重跑 `# 0`，自動複用記憶體中的 20M 資料集（~3 秒切換，防 16 GB 機器 OOM）。
5. **Google Colab + Google Drive 雲端執行與斷線防護（`# 00` & `# 9d`）**：
   - 本機執行自動略過 `# 00`；在 Colab 會自動掛載 Google Drive，將 `MyDrive/chess_numba_data/hce/` 資料集複製到本機 SSD，並把每次 `accept()` / `rollback()` / 候選檔即時備份至 `MyDrive/chess_numba_data/texel_runs/`（支援 `RESUME_FROM_DRIVE = True` 斷線續跑）。
6. **靜態特徵限定模式（`STATIC_FEATURES_ONLY = True`，預設啟用）**：
   - `chess_engine/classical/constants.py` 已還原為 Clean R2 基準（`classical_old/constants.py`）。為避免單純靜態 WDL loss 過度調整與搜尋深度、威脅或王安全平方線強耦合之動態權重，`# 0` 預設開啟 `STATIC_FEATURES_ONLY = True`，透過 `STATIC_ONLY_FROZEN_PARAMS`（47 組）自動凍結 `PASSED_DYNAMICS_*`、`PASSED_PATH_*`、`THREAT_*`、`KING_DANGER_*`、`KING_ATTACK_WEIGHTS`、`SAFE_CHECK_*`、`FLANK_ATTACKS`、`IMBALANCE_*`、`SPACE_*`、`TEMPO_BONUS`，僅調優其餘 66 組靜態特徵（1,049 / 1,276 維）。
   - 全凍結區塊（`1c`、`2a`、`5a`、`5a_grid`、`5a_flank_grid`、`5c`、`7a_div_grid`、`7a`、`7c_grid`）執行時會自動回報 `SKIPPED (static-only)`，且 `accept_if_pass()` 會安全略過；混合區塊（`1b` 僅掃描 `KING_PROX_FRIENDLY_MULT`、`5b` 自動釘死 `FLANK_ATTACKS`）會僅針對靜態項優化。所有 Cell 與參數名稱仍完整保留，以維持參數化覆蓋率測試與手動切換彈性。

### 🚦 Gate Verdict 驗收標準（已升級 per-dataset $K^*$ 分離與殘局／易位子桶守衛）
- `PASS`：`2M unseen holdout` 在其純 WDL $K^*$ 顯著下降（`delta < 0` 且 $z \le 1.0$）、`broad-EG (phase<=64)` 未惡化（$z \le 1.0$）且 `pure king+pawns` / `phase=0` 子桶穩定；若啟用 `val_pgn` / `val_zurichess`，各獨立資料集於自身 $K^*$ 無退步且 `pure king+pawns` 與 `castling!=0` 穩定。
- `CAUTION`：主驗證集改善，但 holdout（$1.0 < z \le 2.0$）、`broad-EG`（$1.0 < z \le 1.8$）、`pure king+pawns` / `phase=0` / `castling!=0` 或獨立驗證集微幅偏正。
- `REJECT`：Holdout 在其 $K^*$ 退步、`broad-EG` 顯著惡化（$z > 1.8$）、holdout `pure king+pawns` / `phase=0` 顯著崩壞（$z > 3.5$）、獨立驗證集整體顯著惡化（$z > 2.0\sim2.5$），或獨立集 `pure king+pawns` / `castling!=0` 極端退步（$z > 4.0$ 且 `delta > 2.0e-5`）→ **自動跳過不套用**。

In [2]:
# =====================================================================
# 檢視 Colab 雲端硬體規格與 CPU 核心數
# =====================================================================
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"
!free -h
!python3 --version


CPU(s):                                  8
On-line CPU(s) list:                     0-7
Model name:                              AMD EPYC 7B12
Thread(s) per core:                      2
Socket(s):                               1
NUMA node(s):                            1
NUMA node0 CPU(s):                       0-7
               total        used        free      shared  buff/cache   available
Mem:            50Gi       1.4Gi        47Gi       2.2Mi       2.7Gi        49Gi
Swap:             0B          0B          0B
Python 3.13.18


In [3]:
# 00. 【Colab / Google Drive 環境初始化】（本機執行會自動略過，直接保留本機路徑與資料）
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = ("google.colab" in sys.modules) or (os.name != "nt" and Path("/content").is_dir())

# ==============================================================================
# Colab / Google Drive 設定（僅在 Google Colab 環境生效）
# ==============================================================================
REPO_URL = "https://github.com/b08203031/chess_numba.git"
BRANCH = "main"
WORKDIR = Path("/content/chess_numba")

# Google Drive 資料目錄：請先手動將 dataset_epd_wdl_20m.npz 與 val_pgn.npz 上傳到此資料夾
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/chess_numba_data/hce")
# 調參產物 (runs / backups / 最新 constants.py) 會即時備份至此 Drive 目錄，防斷線遺失
DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/chess_numba_data/texel_runs_static_sf20")
# 若 Colab 中途斷線，重跑想接續 Drive 上最後一次 Accept 的 constants.py，請改為 True
RESUME_FROM_DRIVE = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    print("=== Colab 硬體與記憶體規格 ===")
    subprocess.run(["bash", "-lc", "lscpu | grep -E 'Model name|CPU\\(s\\)' || nproc"], check=False)
    subprocess.run(["free", "-h"], check=False)

    # 1) Clone 或更新 GitHub 程式碼（不用 git clean -fd，避免誤刪已複製的資料與快取）
    if not (WORKDIR / ".git").is_dir():
        print(f"\n[Git] Cloning {REPO_URL} (branch={BRANCH}) -> {WORKDIR} ...")
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(WORKDIR)], check=True)
    else:
        print(f"\n[Git] Syncing {WORKDIR} -> origin/{BRANCH} ...")
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "checkout", "-f", BRANCH], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)

    os.chdir(WORKDIR)

    # 2) 僅安裝 Colab 缺少的必要相依套件（保留預裝相容之 NumPy / Numba / SciPy 組合）
    missing_pkgs = [
        pkg
        for mod, pkg in (
            ("numba", "numba"),
            ("numpy", "numpy"),
            ("scipy", "scipy"),
            ("chess", "chess"),
        )
        if importlib.util.find_spec(mod) is None
    ]
    if missing_pkgs:
        print(f"[Pip] Installing missing packages: {missing_pkgs}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs], check=True)

    # 3) 檢查 / 還原斷線續跑備份
    DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    if RESUME_FROM_DRIVE:
        restored = []
        for eng in ("classical", "classical_old"):
            saved = DRIVE_BACKUP_DIR / f"{eng}_constants.py"
            dst_c = WORKDIR / "chess_engine" / eng / "constants.py"
            if saved.is_file():
                shutil.copy2(saved, dst_c)
                restored.append(eng)
                print(f"[Resume] 已從 Drive 還原 {eng}/constants.py <- {saved}")
        # 同步先前的 runs 子目錄（含 backups/ 與已產生的 block_*.py）
        for sub in DRIVE_BACKUP_DIR.iterdir():
            if sub.is_dir():
                dst_run = WORKDIR / "tuner" / "runs" / sub.name
                shutil.copytree(sub, dst_run, dirs_exist_ok=True)
                print(f"[Resume] 已還原歷史調參紀錄 -> {dst_run}")
        if restored:
            subprocess.run([sys.executable, "-m", "tuner.sync_constants_snapshot"], check=True)
        if "classical_old" in restored:
            subprocess.run(
                [
                    sys.executable,
                    "-c",
                    "from pathlib import Path; "
                    "from tuner.sync_constants_snapshot import clear_classical_numba_cache; "
                    "clear_classical_numba_cache(Path('chess_engine/classical_old/__pycache__'))",
                ],
                check=True,
            )
    else:
        for eng in ("classical", "classical_old"):
            saved = DRIVE_BACKUP_DIR / f"{eng}_constants.py"
            if saved.is_file():
                print(
                    f"[Info] 偵測到 Drive 上已有備份：{saved}\n"
                    "       目前 RESUME_FROM_DRIVE=False（將從 GitHub 最新基準重新起跑）；"
                    "若要從上次斷線處續跑，請改設 RESUME_FROM_DRIVE=True 後重跑本 Cell。"
                )

    # 4) 從 Google Drive 複製資料集到 Colab 本機 SSD（比直接從 Drive FUSE 讀取快 ~10x）
    local_data_dir = WORKDIR / "tuner" / "data" / "hce"
    local_data_dir.mkdir(parents=True, exist_ok=True)
    if not DRIVE_DATA_DIR.is_dir():
        DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)
        raise FileNotFoundError(
            f"找不到 Drive 資料夾或尚未上傳檔案：{DRIVE_DATA_DIR}\n"
            "請先手動將本機 tuner/data/hce/ 中的檔案上傳至 Google Drive 'MyDrive/chess_numba_data/hce/'：\n"
            "  1) dataset_epd_wdl_20m.npz（必備，~674 MB）\n"
            "  2) val_pgn.npz（開 USE_PGN_VAL=True 時必備，~22.7 MB）\n"
            "  3) val_zurichess.npz / val_pgn.sf19.npz / val_zurichess.sf19.npz / dataset_epd_wdl_20m.sf19.npz（選用）"
        )

    data_files = [
        ("dataset_epd_wdl_20m.npz", True),
        ("val_pgn.npz", False),
        ("val_zurichess.npz", False),
        ("dataset_epd_wdl_20m.sf19.npz", False),
        ("val_pgn.sf19.npz", False),
        ("val_zurichess.sf19.npz", False),
    ]
    for fname, required in data_files:
        src = DRIVE_DATA_DIR / fname
        dst = local_data_dir / fname
        if src.is_file():
            if dst.is_file() and dst.stat().st_size == src.stat().st_size:
                print(f"[Data] 本機 SSD 已存在最新 {fname} ({dst.stat().st_size / (1024**2):.1f} MB)，略過複製。")
            else:
                print(f"[Data] 正在從 Drive 複製 {fname} ({src.stat().st_size / (1024**2):.1f} MB) -> 本機 SSD ...")
                shutil.copy2(src, dst)
        elif required:
            raise FileNotFoundError(
                f"Google Drive 缺少必備訓練檔：{src}\n"
                "請先手動將本機 tuner/data/hce/dataset_epd_wdl_20m.npz 上傳至 Drive 'MyDrive/chess_numba_data/hce/'。"
            )
        elif fname == "val_pgn.npz":
            print("[WARN] Drive 中未找到 val_pgn.npz；若下方要設 USE_PGN_VAL=True 請先手動上傳。")
        else:
            print(f"[Data] 選用檔 {fname} 未提供（未啟用該功能時可略過）。")

    print(f"\n✅ Colab + Google Drive 環境與資料就緒！即時備份目錄：{DRIVE_BACKUP_DIR}")
else:
    DRIVE_BACKUP_DIR = None
    print("✅ 偵測為本機環境，略過 Colab/Drive 掛載（直接使用本機 tuner/data/hce/）。")

Mounted at /content/drive
=== Colab 硬體與記憶體規格 ===

[Git] Cloning https://github.com/b08203031/chess_numba.git (branch=main) -> /content/chess_numba ...
[Pip] Installing missing packages: ['chess']
[Data] 正在從 Drive 複製 dataset_epd_wdl_20m.npz (674.2 MB) -> 本機 SSD ...
[Data] 正在從 Drive 複製 val_pgn.npz (22.7 MB) -> 本機 SSD ...
[Data] 正在從 Drive 複製 val_zurichess.npz (32.5 MB) -> 本機 SSD ...
[Data] 正在從 Drive 複製 dataset_epd_wdl_20m.sf19.npz (58.1 MB) -> 本機 SSD ...
[Data] 正在從 Drive 複製 val_pgn.sf19.npz (4.2 MB) -> 本機 SSD ...
[Data] 正在從 Drive 複製 val_zurichess.sf19.npz (3.4 MB) -> 本機 SSD ...

✅ Colab + Google Drive 環境與資料就緒！即時備份目錄：/content/drive/MyDrive/chess_numba_data/texel_runs_static_sf20


In [4]:
# 0. 環境檢查與載入 TexelTuningSession（首次載入約 25~45 秒；切換模式重跑本 Cell 僅需 ~3 秒）
import os
import sys
from pathlib import Path

# 自動定位專案根目錄（相容本機與 Colab）
if Path("chess_engine").is_dir() and Path("tuner").is_dir():
    REPO_ROOT = Path.cwd()
elif (Path.cwd().parent / "chess_engine").is_dir() and Path.cwd().name == "tuner":
    REPO_ROOT = Path.cwd().parent
    os.chdir(REPO_ROOT)
elif Path("/content/chess_numba").is_dir():
    REPO_ROOT = Path("/content/chess_numba")
    os.chdir(REPO_ROOT)
else:
    raise RuntimeError(f"請在專案根目錄啟動 Notebook，目前目錄：{Path.cwd()}")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tuner.notebook_session import TexelTuningSession

TRAIN_NPZ = REPO_ROOT / "tuner" / "data" / "hce" / "dataset_epd_wdl_20m.npz"
PGN_VAL_NPZ = REPO_ROOT / "tuner" / "data" / "hce" / "val_pgn.npz"
ZURICHESS_VAL_NPZ = REPO_ROOT / "tuner" / "data" / "hce" / "val_zurichess.npz"

# ==============================================================================
# 實驗模式參數（切換任一參數後直接重跑本 Cell，自動複用已載入的 20M 資料集，約 3 秒完成）
# ==============================================================================
# 1) 獨立驗證集選擇
USE_PGN_VAL = True            # True = 啟用 2M holdout + val_pgn.npz 雙重閘門；False = 僅用 2M unseen holdout
USE_ZURICHESS_VAL = True      # True = 啟用 val_zurichess.npz 第三仲裁/tie-breaker 集（若有 val_zurichess.sf19.npz 會自動加印純 WDL sf19-quiet 診斷）

# 2) 訓練寫入目標引擎："classical"     -> chess_engine/classical/constants.py
#                    "classical_old" -> chess_engine/classical_old/constants.py
TARGET_ENGINE = "classical"

# 3) SF19 搜尋評估 + WDL 線性混合標籤：y_mix = (1 - SF_LAMBDA)*WDL + SF_LAMBDA*sigmoid(K_sf * sf_cp)
#    啟用前請先於終端機產生側車標籤檔（支援 Ctrl+C 續傳；可加 --limit 3000000 先跑快速前導實驗）：
#      python -m tuner.data_pipeline.annotate_sf_labels --nodes 4000 --skip-holdout
USE_SF_LABELS = True         # 預設以 Clean R2 基準（classical_old）起跑純 WDL 靜態特徵調優（若存在 .sf19.npz，Holdout/Zurichess 仍自動顯示純 WDL sf19-quiet 診斷）
SF_LAMBDA = 0.20              # 推薦 0.10 ~ 0.15（>0.20 易在獨立 PGN/靜態驗證上過擬合搜尋耦合；上限 <= 0.25；Holdout 與 val_pgn 永遠維持純 WDL）
SF_MAX_TACTICAL_GAP = 180     # |sf_cp - sf_d1_cp| > 180cp 的非平靜戰術局面自動排除出訓練梯度

# 4) 靜態特徵限定模式：True = 凍結 47 組非靜態/搜尋耦合權重，僅調 66 組純靜態特徵（全凍結區塊自動 SKIPPED）
#                      False = 開放全部 113 組參數（高耦合動態項建議搭配對戰/Match-SPSA 驗證）
STATIC_FEATURES_ONLY = True

# 5) SPSA 調參隨機種子：True = 每次 run_block 自動依時間動態產生全新種子（探索不同隨機路徑，跳脫局部死局）；False = 固定各 Cell 預設種子
RANDOM_SEED = True

# "auto": 若本機可用實體 RAM < 4.8 GB 自動取 10M train + 完整 1M dev + 2M holdout 防 Swap；設 None 強制載滿 17M train
MAX_TRAIN_ROWS = "auto"

if (
    "session" in globals()
    and getattr(session, "train_path", None) == TRAIN_NPZ
    and hasattr(session, "train_data")
):
    session.__class__ = TexelTuningSession
    session.reconfigure(
        use_pgn_val=USE_PGN_VAL,
        use_zurichess_val=USE_ZURICHESS_VAL,
        zurichess_val_npz=ZURICHESS_VAL_NPZ,
        target_engine=TARGET_ENGINE,
        pgn_val_npz=PGN_VAL_NPZ,
        use_sf_labels=USE_SF_LABELS,
        sf_lambda=SF_LAMBDA,
        sf_max_tactical_gap=SF_MAX_TACTICAL_GAP,
        drive_backup_dir=globals().get("DRIVE_BACKUP_DIR", None),
        random_seed=RANDOM_SEED,
        static_features_only=STATIC_FEATURES_ONLY,
    )
else:
    session = TexelTuningSession(
        train_npz=TRAIN_NPZ,
        pgn_val_npz=PGN_VAL_NPZ,
        eval_sample=1_000_000,     # SPSA in-run dev / early-stop / K* 校準
        holdout_sample=2_000_000,  # 完全未見的訓練來源 holdout (永遠純 WDL)
        max_train_rows=MAX_TRAIN_ROWS,
        chunk_size=500_000,
        use_pgn_val=USE_PGN_VAL,
        use_zurichess_val=USE_ZURICHESS_VAL,
        zurichess_val_npz=ZURICHESS_VAL_NPZ,
        target_engine=TARGET_ENGINE,
        use_sf_labels=USE_SF_LABELS,
        sf_lambda=SF_LAMBDA,
        sf_max_tactical_gap=SF_MAX_TACTICAL_GAP,
        drive_backup_dir=globals().get("DRIVE_BACKUP_DIR", None),
        random_seed=RANDOM_SEED,
        static_features_only=STATIC_FEATURES_ONLY,
    )
# 開啟「全自動無腦運行模式」：連續執行全部 Cell 時，僅當 Gate 判定為 PASS 才套用變更至目標 constants.py
session.require_pass_to_accept = True
print(
    f"[GUARD] 自動守衛已啟用：require_pass=True | target={session.target_engine!r} | "
    f"static_only={session.static_features_only} | "
    f"use_pgn_val={session.use_pgn_val} | use_zurichess_val={session.use_zurichess_val} | random_seed={session.random_seed} | "
    f"use_sf_labels={session.use_sf_labels} (lambda={session.sf_lambda:.2f}) | "
    f"dev_K*={session._step_dev_k_star:.3f} | hold_K*={session._step_holdout_k_star:.3f}"
)

[1/4] Loading primary dataset: dataset_epd_wdl_20m.npz (RAM before: 49.34 GB free / 50.99 GB total)
      [SF19] Loading sidecar search scores: dataset_epd_wdl_20m.sf19.npz
      [SF19 Blend] lambda=0.20 | calibrated K_sf=0.747 (SF-vs-WDL MSE=0.096669) | active_train=16,627,749/16,627,749 (tactical_skipped=2,066,794) | active_dev=1,000,000 | holdout=pure_WDL
      [SF19-diag] Holdout pure-WDL split (dataset_epd_wdl_20m.sf19.npz): annotated=2,000,000/2,000,000 (quiet=1,789,782, tactical=210,218)
      Loaded 19,627,749 positions -> active_train=16,627,749 | active_dev=1,000,000 | unseen_holdout(WDL)=2,000,000 (RAM now: 45.66 GB free / 50.99 GB total)
[2/4] Loading independent PGN validation set: val_pgn.npz
      Loaded 1,593,187 unique PGN positions (castling!=0: 104,249 [6.54%], RAM now: 45.42 GB free / 50.99 GB total)
      [SF19-diag] PGN pure-WDL split (val_pgn.sf19.npz): annotated=1,593,187/1,593,187 (quiet=1,556,972, tactical=36,215)
[2b/4] Loading Zurichess quiet EPD tie-breaker

In [5]:
# 0b. 檢視目前 Baseline 在兩個獨立驗證集上的健康度與最佳 K*
baseline_stats = session.baseline_report()

=== Current Step Baseline Health Report [target=classical -> chess_engine/classical/constants.py | use_pgn_val=True | train=SF19+WDL(lambda=0.20) | dev_K*=1.155 | hold_K*=1.239] ===
  [1] dataset_epd_wdl_20m.npz unseen holdout [pure WDL] (n=2,000,000):
      MSE @ K*=1.239: 0.0915388
      Pure-WDL split  : sf19-quiet=0.0918378 (n=1,789,782) | sf19-tactical=0.0889925 (n=210,218)
  [2] val_pgn.npz independent PGN val (n=1,593,187):
      Static Eval MSE @ K*=1.148: 0.0567730
      Castling!=0 MSE @ K*      : 0.1060729 (n=104,249)
      Pure-WDL split  : sf19-quiet=0.0570249 (n=1,556,972) | sf19-tactical=0.0459441 (n=36,215)
      300k Search Ceiling @ K*=1.477: 0.0445078 (gap=+0.0122652)
  [3] val_zurichess.npz Zurichess quiet EPD (n=1,079,507):
      Static Eval MSE @ K*=1.396: 0.0487617
      Pure-WDL split  : sf19-quiet=0.0460237 (n=936,437) | sf19-tactical=0.0666826 (n=143,070)


In [6]:
# 0c. 熱重載調參器核心（若修改過 tuner/*.py，無須重啟 Kernel 或重載 20M 資料集即可生效）
import importlib
import tuner.data_pipeline.annotate_sf_labels as _sf_mod
import tuner.parameters as _params_mod
import tuner.tuner as _tuner_mod
import tuner.notebook_session as _ns_mod

importlib.reload(_sf_mod)
importlib.reload(_params_mod)
importlib.reload(_tuner_mod)
importlib.reload(_ns_mod)
if "session" in globals():
    session.__class__ = _ns_mod.TexelTuningSession
    session.require_pass_to_accept = True
print(
    "✅ 已熱重載最新 Scale-Aware SPSA、自動 K* 對齊、SF19 混合標籤與 Gate 判準！"
    + (
        f" [target={session.target_engine!r}, static_only={getattr(session, 'static_features_only', True)}, "
        f"use_pgn_val={session.use_pgn_val}, use_sf_labels={getattr(session, 'use_sf_labels', False)}]"
        if "session" in globals()
        else ""
    )
)

✅ 已熱重載最新 Scale-Aware SPSA、自動 K* 對齊、SF19 混合標籤與 Gate 判準！ [target='classical', static_only=True, use_pgn_val=True, use_sf_labels=True]


---
## Block 1：通路兵系統（Passed Pawns — 對應 `ORD-1..3` 與 `KP-DISCONT` 修復後最優先）
`ORD-1..3` 修正了通路兵路徑安全與攻擊遮罩計算順序，`KP-DISCONT` 補齊多兵純王兵殘局縮放：
- **Block 1a**：連續 centipawn 表（`PASSED_PAWN_BONUS`、`PASSED_FILE_BONUS`、`PASSED_CONNECTED_BONUS`、`PASSED_PHALANX_BONUS`），自動在基準校準 $K^*$ 下以 `alpha=35000, A=150, batch_size=65536` 調優，防止假性尺度膨脹。
- **Block 1b**：離散動態公式係數（`PASSED_DYNAMICS_MULT=3:5`、`PASSED_DYNAMICS_OFFSET=-12:-8` 置中於 `-10`、`KING_PROX_FRIENDLY_MULT=1:2`）；在 `STATIC_FEATURES_ONLY=True` 下僅掃描靜態 `KING_PROX_FRIENDLY_MULT`（凍結 `PASSED_DYNAMICS_*`）。
- **Block 1c**：路徑安全係數（`PASSED_PATH_*`，Clean R2=`10/6/2/3`，各 $\theta_0 \pm 1$ 共 $3^4 = 81$ 組）離散窮舉；在 `STATIC_FEATURES_ONLY=True` 下屬全凍結區塊會自動 `SKIPPED`。
- **Block 1d**：殘局王距離權重與上下界（`KING_PROX_*`），以 `c=1.0, c_floor=1.0, alpha=9000, A=200, batch_size=98304` 降噪緩步對齊緊邊界整數格點（避免 `scales` 放大導致前 50 步過衝）。
- **Block 1e_grid**：多兵純王兵殘局近王活動倍率（`KING_PAWN_PROXIMITY_FACTOR=2:8`）離散窮舉掃描。

In [7]:
# [Run] Block 1a — 通路兵主獎勵表 (CP Tables, dev_K* 對齊與穩定步長)
res_1a = session.run_block(
    "block1a_passed_tables",
    tune=[
        "PASSED_PAWN_BONUS",
        "PASSED_FILE_BONUS",
        "PASSED_CONNECTED_BONUS",
        "PASSED_PHALANX_BONUS",
    ],
    iters=3500,
    c=2.0,
    alpha=35000.0,
    A=150.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=(0 if getattr(session, "random_seed", False) else 101),
)

  [SPSA Seed] Using dynamic time seed: 1681112754
=== Running Block [block1a_passed_tables] -> tuner/runs/classical_with_pgn_sf20/block1a_passed_tables.py (K=1.155, labels=SF_mix(lambda=0.20)) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1213 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.5004, std=0.3466
  Continuous/mixed labels detected (1001 unique values).
Freezing all parameters except: ['PASSED_PAWN_BONUS', 'PASSED_FILE_BONUS', 'PASSED_CONNECTED_BONUS', 'PASSED_PHALANX_BONUS']
Active parameters: 26 / 1276
Coordinate auto-scale active: min=1.00, median=1.00, max=4.00
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.155): 0.0648910
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0392448  n=74,346
    phase=1..31           0.0543674  n=38,2

In [8]:
# [Accept] Block 1a — 僅在 Gate 判定為 PASS 時自動套用，非 PASS 自動略過
session.accept_if_pass("block1a_passed_tables")


[SKIP] [block1a_passed_tables] Gate 判定為 'CAUTION' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

In [9]:
# [Run] Block 1b — 通路兵動態倍率離散窮舉 (Clean R2 MULT=4, OFFSET=-10；STATIC_FEATURES_ONLY=True 僅掃描 KING_PROX_FRIENDLY_MULT)
res_1b = session.scan_grid(
    "block1b_passed_dynamics_grid",
    grids=[
        "PASSED_DYNAMICS_MULT=3:5",
        "PASSED_DYNAMICS_OFFSET=-12:-8",
        "KING_PROX_FRIENDLY_MULT=1:2",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=10,
)

  [Static-Only] Grid [block1b_passed_dynamics_grid] freezing non-static scalar(s): PASSED_DYNAMICS_MULT, PASSED_DYNAMICS_OFFSET
=== Discrete Grid Scan [block1b_passed_dynamics_grid] -> tuner/runs/classical_with_pgn_sf20/block1b_passed_dynamics_grid.py (K=1.155) ===
Constraints configured: 1213 constraints active.
Scan sample (dev): 1,000,000 (K=1.155); excluded fixed validation=0
  evaluated   1/2: (1,)
  evaluated   2/2: (2,)
Candidates sorted by overall delta (negative is better):
  KING_PROX_FRIENDLY_MULT          all     broad-EG      phase=0    queenless     minor-EG      rook-EG
             1 +0.00000e+00 +0.00000e+00 +0.00000e+00 +0.00000e+00 +0.00000e+00 +0.00000e+00
             2 +2.62292e-04 +4.31417e-04 +6.15965e-04 +5.18576e-04 +4.44281e-04 +5.33938e-04

Best discrete grid point on dev: {'KING_PROX_FRIENDLY_MULT': 1}
--- Dual Unseen Validation for [block1b_passed_dynamics_grid] ---
Effective integer changes:
  (none)

[dataset_epd_wdl_20m.npz [unseen holdout, pure WDL]] P

In [10]:
# [Accept] Block 1b — 若離散掃描在 holdout 與 val_pgn 皆穩定改善再套用
session.accept_if_pass("block1b_passed_dynamics_grid")


[SKIP] [block1b_passed_dynamics_grid] Gate 判定為 'NO-OP (0 integer changes)' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

In [11]:
# [Run] Block 1c — 通路兵路徑安全係數精確窮舉 (Clean R2=10/6/2/3，3^4 = 81 組；STATIC_FEATURES_ONLY=True 自動 SKIPPED)
res_1c = session.scan_grid(
    "block1c_passed_path_grid",
    grids=[
        "PASSED_PATH_SAFE_NONE_ATTACK=9:11",
        "PASSED_PATH_SAFE_EDGE_ATTACK=5:7",
        "PASSED_PATH_SAFE_BLOCK_ONLY=1:3",
        "PASSED_PATH_SUPPORT_BONUS=2:4",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=10,
)

=== Skipping Grid [block1c_passed_path_grid] (static_features_only=True; non-static scalars frozen: PASSED_PATH_SAFE_NONE_ATTACK, PASSED_PATH_SAFE_EDGE_ATTACK, PASSED_PATH_SAFE_BLOCK_ONLY, PASSED_PATH_SUPPORT_BONUS) ===


In [12]:
# [Accept] Block 1c
session.accept_if_pass("block1c_passed_path_grid")


[SKIP] [block1c_passed_path_grid] Gate 判定為 'SKIPPED (static-only)' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

In [13]:
# [Run] Block 1d — 通路兵王距離權重與夾限微調 (c=1.0, alpha=9000, A=200, batch=98304 防前 50 步過衝)
res_1d = session.run_block(
    "block1d_king_prox",
    tune=[
        "KING_PROXIMITY_MAX_DIST",
        "KING_PROX_ENEMY_MULT",
        "KING_PROX_ENEMY_DIV",
        "KING_PROX_MIN_BONUS",
        "KING_PROX_MAX_BONUS",
    ],
    iters=2500,
    c=1.0,
    c_floor=1.0,
    alpha=9000.0,
    A=200.0,
    batch_size=98304,
    report_every=50,
    patience=25,
    seed=(0 if getattr(session, "random_seed", False) else 104),
)

  [SPSA Seed] Using dynamic time seed: 1681414254
=== Running Block [block1d_king_prox] -> tuner/runs/classical_with_pgn_sf20/block1d_king_prox.py (K=1.155, labels=SF_mix(lambda=0.20)) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1213 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.5004, std=0.3466
  Continuous/mixed labels detected (1001 unique values).
Freezing all parameters except: ['KING_PROXIMITY_MAX_DIST', 'KING_PROX_ENEMY_MULT', 'KING_PROX_ENEMY_DIV', 'KING_PROX_MIN_BONUS', 'KING_PROX_MAX_BONUS']
Active parameters: 5 / 1276
Coordinate auto-scale active: min=1.00, median=1.00, max=2.73
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.155): 0.0648910
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0392448  n=74,346
    phase=1..31         

In [14]:
# [Accept] Block 1d
session.accept_if_pass("block1d_king_prox")


[SKIP] [block1d_king_prox] Gate 判定為 'CAUTION' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

In [15]:
# [Run] Block 1e_grid — 王兵純殘局近王活動係數離散窮舉
res_1e = session.scan_grid(
    "block1e_kp_prox_grid",
    grids=[
        "KING_PAWN_PROXIMITY_FACTOR=2:8",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=10,
)

=== Discrete Grid Scan [block1e_kp_prox_grid] -> tuner/runs/classical_with_pgn_sf20/block1e_kp_prox_grid.py (K=1.155) ===
Constraints configured: 1213 constraints active.
Scan sample (dev): 1,000,000 (K=1.155); excluded fixed validation=0
  evaluated   1/7: (2,)
  evaluated   2/7: (3,)
  evaluated   3/7: (4,)
  evaluated   4/7: (5,)
  evaluated   5/7: (6,)
  evaluated   6/7: (7,)
  evaluated   7/7: (8,)
Candidates sorted by overall delta (negative is better):
  KING_PAWN_PROXIMITY_FACTOR          all     broad-EG      phase=0    queenless     minor-EG      rook-EG
             3 +0.00000e+00 +0.00000e+00 +0.00000e+00 +0.00000e+00 +0.00000e+00 +0.00000e+00
             2 +1.06139e-06 +1.81959e-06 +3.57049e-06 +2.36041e-06 +0.00000e+00 +0.00000e+00
             4 +1.35648e-06 +2.32547e-06 +4.56316e-06 +3.01665e-06 +0.00000e+00 +0.00000e+00
             5 +5.05465e-06 +8.66542e-06 +1.70037e-05 +1.12410e-05 +0.00000e+00 +0.00000e+00
             6 +1.08130e-05 +1.85372e-05 +3.63747e-05 +2.

In [16]:
# [Accept] Block 1e_grid
session.accept_if_pass("block1e_kp_prox_grid")


[SKIP] [block1e_kp_prox_grid] Gate 判定為 'NO-OP (0 integer changes)' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

---
## Block 2：威脅、前哨與棋子協同（對應 `BUG-5` 威脅遮罩、`BUG-6` 弱后、`BUG-7` 前哨修復後）
- **Block 2a**：威脅評估表（`THREAT_*`，對王項已結構性釘死為 0；在 `STATIC_FEATURES_ONLY=True` 下自動 `SKIPPED`）。
- **Block 2b**：騎士/主教前哨、車開放線、弱后、雙象與輕子協同（`OUTPOST_*`、`ROOK_ON_*`、`WEAK_QUEEN`、`BISHOP_*`、`KING_PROTECTOR`）。
- **Block 2b_grid**：主教同色兵遇到中心受阻兵時的懲罰加重倍率（`BISHOP_PAWNS_CENTER_BLOCKED_FACTOR=0:3`）離散窮舉掃描。

In [17]:
# [Run] Block 2a — 威脅表 (Threats, alpha=45000, A=150；STATIC_FEATURES_ONLY=True 自動 SKIPPED)
res_2a = session.run_block(
    "block2a_threats",
    tune=[
        "THREAT_SAFE_PAWN",
        "THREAT_KNIGHT_ON_QUEEN",
        "THREAT_SLIDER_ON_QUEEN",
        "THREAT_BY_MINOR",
        "THREAT_BY_ROOK",
        "THREAT_BY_KING",
        "THREAT_HANGING",
        "THREAT_RESTRICTED_PIECE",
        "THREAT_PAWN_PUSH",
    ],
    iters=3500,
    c=2.5,
    alpha=45000.0,
    A=150.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=(0 if getattr(session, "random_seed", False) else 201),
)

=== Skipping Block [block2a_threats] (static_features_only=True; non-static params frozen: THREAT_SAFE_PAWN, THREAT_KNIGHT_ON_QUEEN, THREAT_SLIDER_ON_QUEEN, THREAT_BY_MINOR, THREAT_BY_ROOK, THREAT_BY_KING, THREAT_HANGING, THREAT_RESTRICTED_PIECE, THREAT_PAWN_PUSH) ===


In [18]:
# [Accept] Block 2a
session.accept_if_pass("block2a_threats")


[SKIP] [block2a_threats] Gate 判定為 'SKIPPED (static-only)' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

In [19]:
# [Run] Block 2b — 前哨與棋子協同 (Outposts & Piece Coordination, alpha=25000, A=220, batch=131072, iters=4000 防純 WDL 初期過衝)
res_2b = session.run_block(
    "block2b_outposts_coordination",
    tune=[
        "OUTPOST_BONUS_KNIGHT",
        "OUTPOST_BONUS_BISHOP",
        "REACHABLE_OUTPOST_BONUS",
        "MINOR_BEHIND_PAWN",
        "BISHOP_PAWNS_PENALTY",
        "LONG_DIAGONAL_BISHOP",
        "BISHOP_PAIR_BONUS",
        "BISHOP_PAIR_PAWN_SCALE",
        "ROOK_ON_SEMI_OPEN_FILE_BONUS",
        "ROOK_ON_OPEN_FILE_BONUS",
        "ROOK_ON_QUEEN_FILE",
        "WEAK_QUEEN",
        "KING_PROTECTOR",
    ],
    iters=4000,
    c=2.5,
    alpha=25000.0,
    A=220.0,
    batch_size=131072,
    report_every=50,
    patience=35,
    seed=(0 if getattr(session, "random_seed", False) else 202),
)

  [SPSA Seed] Using dynamic time seed: 1681643306
=== Running Block [block2b_outposts_coordination] -> tuner/runs/classical_with_pgn_sf20/block2b_outposts_coordination.py (K=1.155, labels=SF_mix(lambda=0.20)) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1213 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.5004, std=0.3466
  Continuous/mixed labels detected (1001 unique values).
Freezing all parameters except: ['OUTPOST_BONUS_KNIGHT', 'OUTPOST_BONUS_BISHOP', 'REACHABLE_OUTPOST_BONUS', 'MINOR_BEHIND_PAWN', 'BISHOP_PAWNS_PENALTY', 'LONG_DIAGONAL_BISHOP', 'BISHOP_PAIR_BONUS', 'BISHOP_PAIR_PAWN_SCALE', 'ROOK_ON_SEMI_OPEN_FILE_BONUS', 'ROOK_ON_OPEN_FILE_BONUS', 'ROOK_ON_QUEEN_FILE', 'WEAK_QUEEN', 'KING_PROTECTOR']
Active parameters: 42 / 1276
Coordinate auto-scale active: min=1.00, median=1.00, max=1.47
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fi

In [20]:
# [Accept] Block 2b
session.accept_if_pass("block2b_outposts_coordination")


[SKIP] [block2b_outposts_coordination] Gate 判定為 'NO-OP (0 integer changes)' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

In [21]:
# [Run] Block 2b_grid — 主教同色兵中心受阻懲罰倍率離散掃描
res_2b_grid = session.scan_grid(
    "block2b_bishop_pawns_factor_grid",
    grids=[
        "BISHOP_PAWNS_CENTER_BLOCKED_FACTOR=0:3",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=10,
)

=== Discrete Grid Scan [block2b_bishop_pawns_factor_grid] -> tuner/runs/classical_with_pgn_sf20/block2b_bishop_pawns_factor_grid.py (K=1.155) ===
Constraints configured: 1213 constraints active.
Scan sample (dev): 1,000,000 (K=1.155); excluded fixed validation=0
  evaluated   1/4: (0,)
  evaluated   2/4: (1,)
  evaluated   3/4: (2,)
  evaluated   4/4: (3,)
Candidates sorted by overall delta (negative is better):
  BISHOP_PAWNS_CENTER_BLOCKED_FACTOR          all     broad-EG      phase=0    queenless     minor-EG      rook-EG
             1 +0.00000e+00 +0.00000e+00 +0.00000e+00 +0.00000e+00 +0.00000e+00 +0.00000e+00
             0 +4.91813e-05 +6.18816e-05 +5.67288e-05 +5.93798e-05 +1.01091e-04 +4.97771e-05
             2 +8.45698e-05 +1.06521e-04 +5.60733e-05 +1.28366e-04 +9.02014e-05 +1.42036e-04
             3 +2.94289e-04 +3.67854e-04 +2.11858e-04 +4.25085e-04 +3.60779e-04 +4.54404e-04

Best discrete grid point on dev: {'BISHOP_PAWNS_CENTER_BLOCKED_FACTOR': 1}
--- Dual Unseen Valid

In [22]:
# [Accept] Block 2b_grid
session.accept_if_pass("block2b_bishop_pawns_factor_grid")


[SKIP] [block2b_bishop_pawns_factor_grid] Gate 判定為 'NO-OP (0 integer changes)' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

---
## Block 3：機動性查表（Mobility Tables — 對應 `BUG-1` 斜向釘死幾何修復後）
每張 Mobility 表的 `0-mobility` (MG/EG) 已錨定於 `theta0`（避免與棋子基礎材質產生常數偏移共線性），並強制單調遞增約束。

In [23]:
# [Run] Block 3 — 四子機動性查表 (N / B / R / Q Mobility, iters=5000, batch=98304)
res_3 = session.run_block(
    "block3_mobility",
    tune=[
        "KNIGHT_MOBILITY_BONUS",
        "BISHOP_MOBILITY_BONUS",
        "ROOK_MOBILITY_BONUS",
        "QUEEN_MOBILITY_BONUS",
    ],
    iters=5000,
    c=2.5,
    alpha=45000.0,
    A=150.0,
    batch_size=98304,
    report_every=50,
    patience=35,
    seed=(0 if getattr(session, "random_seed", False) else 301),
)

  [SPSA Seed] Using dynamic time seed: 1682002282
=== Running Block [block3_mobility] -> tuner/runs/classical_with_pgn_sf20/block3_mobility.py (K=1.155, labels=SF_mix(lambda=0.20)) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1213 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.5004, std=0.3466
  Continuous/mixed labels detected (1001 unique values).
Freezing all parameters except: ['KNIGHT_MOBILITY_BONUS', 'BISHOP_MOBILITY_BONUS', 'ROOK_MOBILITY_BONUS', 'QUEEN_MOBILITY_BONUS']
Active parameters: 124 / 1276
Coordinate auto-scale active: min=1.00, median=1.22, max=3.13
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.155): 0.0648910
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0392448  n=74,346
    phase=1..31           0.0543674  n=38,267
   

In [24]:
# [Accept] Block 3
session.accept_if_pass("block3_mobility")


[PASS] [block3_mobility] Gate 判定為 PASS -> 正在套用變更至 chess_engine/classical/constants.py 並建立備份...
  KNIGHT_MOBILITY_BONUS: [-17, -26, -12, -19, 1, -9] -> [-17, -26, -12, -19, 0, -9]
  BISHOP_MOBILITY_BONUS: [-10, -18, 5, -13, 15, 0] -> [-10, -18, 5, -14, 16, -1]
  ROOK_MOBILITY_BONUS: [-27, -32, -1, -1, 14, 20] -> [-27, -32, 0, -1, 15, 20]
  QUEEN_MOBILITY_BONUS: [-26, -10, -13, -4, 8, 7] -> [-26, -10, -13, -4, 8, 7]
Summary: 4 numeric change(s), 109 identical (skipped), 113 names in tuned dump.
Updated 4 constant(s) in chess_engine/classical/constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Wrote tuner/constants_snapshot.py from chess_engine/classical/constants.py; removed 85 stale classical Numba cache file(s)
[block3_mobility] Accepted 4 parameter group(s) into chess_engine/classical/constants.py. Backup saved to tuner/runs/classical_with_pgn_sf20/backups/20261116_122051_classical_before_block3_mobility_constants.py.
[D

['KNIGHT_MOBILITY_BONUS',
 'BISHOP_MOBILITY_BONUS',
 'ROOK_MOBILITY_BONUS',
 'QUEEN_MOBILITY_BONUS']

---
## Block 4：兵型結構與連兵（Pawn Structure & Connected Pawns）
- **Block 4a**：孤兵、疊兵、落後兵、弱兵懲罰與連兵等級獎勵表。
- **Block 4b**：連兵支撐乘數（`CONNECTED_SUPPORT_WEIGHT`、`CONNECTED_SUPPORT_WEIGHT_EG`）離散窮舉。

In [25]:
# [Run] Block 4a — 兵型懲罰與連兵獎勵表 (iters=4000, alpha=40000, A=180, batch=131072 防稀疏 K+P 梯度雜訊)
res_4a = session.run_block(
    "block4a_pawn_structure",
    tune=[
        "ISOLATED_PAWN_PENALTY",
        "DOUBLED_PAWN_PENALTY",
        "BACKWARD_PAWN_PENALTY",
        "WEAK_LEVER_PENALTY",
        "WEAK_UNOPPOSED_PENALTY",
        "CONNECTED_BONUS",
        "CONNECTED_BONUS_EG",
    ],
    iters=4000,
    c=2.0,
    alpha=40000.0,
    A=180.0,
    batch_size=131072,
    report_every=50,
    patience=35,
    seed=(0 if getattr(session, "random_seed", False) else 401),
)

  [SPSA Seed] Using dynamic time seed: 1682818168
=== Running Block [block4a_pawn_structure] -> tuner/runs/classical_with_pgn_sf20/block4a_pawn_structure.py (K=1.155, labels=SF_mix(lambda=0.20)) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1213 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.5004, std=0.3466
  Continuous/mixed labels detected (1001 unique values).
Freezing all parameters except: ['ISOLATED_PAWN_PENALTY', 'DOUBLED_PAWN_PENALTY', 'BACKWARD_PAWN_PENALTY', 'WEAK_LEVER_PENALTY', 'WEAK_UNOPPOSED_PENALTY', 'CONNECTED_BONUS', 'CONNECTED_BONUS_EG']
Active parameters: 22 / 1276
Coordinate auto-scale active: min=1.00, median=1.00, max=2.27
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.155): 0.0648870
  Fixed-theta0 validation buckets (MSE, n):
    phase=0     

In [26]:
# [Accept] Block 4a
session.accept_if_pass("block4a_pawn_structure")


[SKIP] [block4a_pawn_structure] Gate 判定為 'REJECT' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

In [27]:
# [Run] Block 4b — 連兵支撐權重離散窮舉 (Clean R2 目前 MG=14, EG=5)
res_4b = session.scan_grid(
    "block4b_connected_support_grid",
    grids=[
        "CONNECTED_SUPPORT_WEIGHT=12:16",
        "CONNECTED_SUPPORT_WEIGHT_EG=3:7",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=9,
)

=== Discrete Grid Scan [block4b_connected_support_grid] -> tuner/runs/classical_with_pgn_sf20/block4b_connected_support_grid.py (K=1.155) ===
Constraints configured: 1213 constraints active.
  [Grid Constraint] CONNECTED_SUPPORT_WEIGHT: clipped [12, 13, 14, 15, 16] -> [13, 14, 15] (bounds=[13, 15])
  [Grid Constraint] CONNECTED_SUPPORT_WEIGHT_EG: clipped [3, 4, 5, 6, 7] -> [4, 5, 6] (bounds=[4, 6])
Scan sample (dev): 1,000,000 (K=1.155); excluded fixed validation=0
  evaluated   1/9: (13, 4)
  evaluated   2/9: (13, 5)
  evaluated   3/9: (13, 6)
  evaluated   4/9: (14, 4)
  evaluated   5/9: (14, 5)
  evaluated   6/9: (14, 6)
  evaluated   7/9: (15, 4)
  evaluated   8/9: (15, 5)
  evaluated   9/9: (15, 6)
Candidates sorted by overall delta (negative is better):
  CONNECTED_SUPPORT_WEIGHT CONNECTED_SUPPORT_WEIGHT_EG          all     broad-EG      phase=0    queenless     minor-EG      rook-EG
            15            4 -2.90984e-06 -5.35445e-06 -3.08069e-06 -3.92359e-06 -6.38832e-06 -3.5

In [28]:
# [Accept] Block 4b
session.accept_if_pass("block4b_connected_support_grid")


[PASS] [block4b_connected_support_grid] Gate 判定為 PASS -> 正在套用變更至 chess_engine/classical/constants.py 並建立備份...
  CONNECTED_SUPPORT_WEIGHT: [14] -> [15]
  CONNECTED_SUPPORT_WEIGHT_EG: [5] -> [4]
Summary: 2 numeric change(s), 111 identical (skipped), 113 names in tuned dump.
Updated 2 constant(s) in chess_engine/classical/constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Wrote tuner/constants_snapshot.py from chess_engine/classical/constants.py; removed 0 stale classical Numba cache file(s)
[block4b_connected_support_grid] Accepted 2 parameter group(s) into chess_engine/classical/constants.py. Backup saved to tuner/runs/classical_with_pgn_sf20/backups/20261116_123347_classical_before_block4b_connected_support_grid_constants.py.
[Drive Backup] (block4b_connected_support_grid_accept) Synced -> /content/drive/MyDrive/chess_numba_data/texel_runs_static_sf20/classical_with_pgn_sf20


['CONNECTED_SUPPORT_WEIGHT', 'CONNECTED_SUPPORT_WEIGHT_EG']

---
## Block 5：王安全、王前兵盾與王 PST（對應 `BUG-8` King Ring 修復後）
依 `tuner/README.md` 規範，**將 King Danger 與 Shelter + 王 PST 分開調優**，防止三者搶同一組王安全訊號：
- **Block 5a**：`kingDanger` 連續權重（`STATIC_FEATURES_ONLY=True` 下自動 `SKIPPED`）。
- **Block 5a_grid**：`KING_DANGER_THRESHOLD` 與 `KING_DANGER_OFFSET` 離散閘門掃描（`STATIC_FEATURES_ONLY=True` 下自動 `SKIPPED`）。
- **Block 5a_flank_grid**：王翼側攻擊/防守與兵盾回饋整數係數離散掃描（`STATIC_FEATURES_ONLY=True` 下自動 `SKIPPED`）。
- **Block 5b**：`SHELTER_*`、`STORM`、`PAWNLESS_FLANK`、`FLANK_ATTACKS`（`STATIC_FEATURES_ONLY=True` 自動釘死）、`TRAPPED_ROOK` 與窄帶王 `PST`（`king_pst_only=True`）。
- **Block 5c**：王安全輸出映射比例微調（`STATIC_FEATURES_ONLY=True` 下自動 `SKIPPED`）。
  > ⚠️ **特別檢查點**：執行 Block 5a/5b/5c 時，請務必確認 `val_pgn.npz` 的 **`castling!=0`** 桶（10.4 萬個真實具易位權局面）沒有顯著惡化！

In [29]:
# [Run] Block 5a — King Danger 攻擊權重與安全將軍係數 (auto_scale=6..15, alpha=30000, A=150；STATIC_FEATURES_ONLY=True 自動 SKIPPED)
res_5a = session.run_block(
    "block5a_king_danger",
    tune=[
        "KING_ATTACK_WEIGHTS",
        "SAFE_CHECK_QUEEN",
        "SAFE_CHECK_ROOK",
        "SAFE_CHECK_BISHOP",
        "SAFE_CHECK_KNIGHT",
        "KING_DANGER_WEAK_SQ",
        "KING_DANGER_UNSAFE_CHECK",
        "KING_DANGER_BLOCKERS",
        "KING_DANGER_ATTACK_ON_KING_SQ",
        "KING_DANGER_NO_QUEEN",
        "KING_DANGER_NO_QUEEN_ROOKLESS",
        "KING_DANGER_KNIGHT_DEF",
    ],
    iters=3500,
    c=2.5,
    alpha=30000.0,
    A=150.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=(0 if getattr(session, "random_seed", False) else 501),
)

=== Skipping Block [block5a_king_danger] (static_features_only=True; non-static params frozen: KING_ATTACK_WEIGHTS, SAFE_CHECK_QUEEN, SAFE_CHECK_ROOK, SAFE_CHECK_BISHOP, SAFE_CHECK_KNIGHT, KING_DANGER_WEAK_SQ, KING_DANGER_UNSAFE_CHECK, KING_DANGER_BLOCKERS, KING_DANGER_ATTACK_ON_KING_SQ, KING_DANGER_NO_QUEEN, KING_DANGER_NO_QUEEN_ROOKLESS, KING_DANGER_KNIGHT_DEF) ===


In [30]:
# [Accept] Block 5a
session.accept_if_pass("block5a_king_danger")


[SKIP] [block5a_king_danger] Gate 判定為 'SKIPPED (static-only)' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

In [31]:
# [Run] Block 5a_grid — King Danger 啟動門檻與偏置離散掃描 (STATIC_FEATURES_ONLY=True 自動 SKIPPED)
res_5a_grid = session.scan_grid(
    "block5a_king_danger_grid",
    grids=[
        "KING_DANGER_OFFSET=0:4",
        "KING_DANGER_THRESHOLD=66:72",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=10,
)

=== Skipping Grid [block5a_king_danger_grid] (static_features_only=True; non-static scalars frozen: KING_DANGER_OFFSET, KING_DANGER_THRESHOLD) ===


In [32]:
# [Accept] Block 5a_grid
session.accept_if_pass("block5a_king_danger_grid")


[SKIP] [block5a_king_danger_grid] Gate 判定為 'SKIPPED (static-only)' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

In [33]:
# [Run] Block 5a_flank_grid — 王翼攻防與兵盾回饋係數離散窮舉 (5^3 = 125 組；STATIC_FEATURES_ONLY=True 自動 SKIPPED)
res_5a_flank = session.scan_grid(
    "block5a_king_flank_grid",
    grids=[
        "KING_FLANK_ATTACK_NUM=1:5",
        "KING_FLANK_DEFENSE_MULT=2:6",
        "KING_SHELTER_FEEDBACK_NUM=4:8",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=10,
)

=== Skipping Grid [block5a_king_flank_grid] (static_features_only=True; non-static scalars frozen: KING_FLANK_ATTACK_NUM, KING_FLANK_DEFENSE_MULT, KING_SHELTER_FEEDBACK_NUM) ===


In [34]:
# [Accept] Block 5a_flank_grid
session.accept_if_pass("block5a_king_flank_grid")


[SKIP] [block5a_king_flank_grid] Gate 判定為 'SKIPPED (static-only)' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

In [35]:
session._reload_parameter_manager()
session._refresh_baseline_score_caches()

Unchanged tuner/constants_snapshot.py


In [36]:
# [Run] Block 5b — 王前兵盾、兵風暴、受困車與窄帶王 PST (alpha=40000, A=200, iters=5500, batch=131072；請重點檢查 val_pgn castling!=0 桶！)
res_5b = session.run_block(
    "block5b_shelter_storm_king_pst",
    tune=[
        "SHELTER_STRENGTH",
        "UNBLOCKED_STORM",
        "BLOCKED_STORM",
        "BLOCKED_STORM_EG",
        "SHELTER_BASE_MG",
        "SHELTER_BASE_EG",
        "KING_PAWN_DIST_PENALTY_EG",
        "PAWNLESS_FLANK",
        "FLANK_ATTACKS",
        "TRAPPED_ROOK",
        "PST_MG",
        "PST_EG",
    ],
    king_pst_only=True,
    iters=5500,
    c=2.5,
    alpha=40000.0,
    A=200.0,
    batch_size=131072,
    report_every=50,
    patience=40,
    seed=(0 if getattr(session, "random_seed", False) else 502),
)

  [Static-Only] Block [block5b_shelter_storm_king_pst] freezing non-static param(s): FLANK_ATTACKS
  [SPSA Seed] Using dynamic time seed: 1683592598
=== Running Block [block5b_shelter_storm_king_pst] -> tuner/runs/classical_with_pgn_sf20/block5b_shelter_storm_king_pst.py (K=1.155, labels=SF_mix(lambda=0.20)) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1213 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.5004, std=0.3466
  Continuous/mixed labels detected (1001 unique values).
PST-piece mode active; also: ['SHELTER_STRENGTH', 'UNBLOCKED_STORM', 'BLOCKED_STORM', 'BLOCKED_STORM_EG', 'SHELTER_BASE_MG', 'SHELTER_BASE_EG', 'KING_PAWN_DIST_PENALTY_EG', 'PAWNLESS_FLANK', 'TRAPPED_ROOK']
PST piece filter: [K] rows active on PST_MG/EG
Active parameters: 193 / 1276
Coordinate auto-scale active: min=1.00, median=1.00, max=4.00
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket 

In [37]:
# [Accept] Block 5b — 確認 val_pgn 的 all 與 castling!=0 桶皆無顯著退步後再套用
session.accept_if_pass("block5b_shelter_storm_king_pst")


[PASS] [block5b_shelter_storm_king_pst] Gate 判定為 PASS -> 正在套用變更至 chess_engine/classical/constants.py 並建立備份...
  PST_MG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
  PST_EG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
  SHELTER_STRENGTH: [15, 56, 59, 45, 30, 20] -> [16, 55, 57, 45, 30, 20]
  UNBLOCKED_STORM: [-69, 237, 86, -61, -41, -30] -> [-67, 235, 83, -59, -41, -30]
  BLOCKED_STORM: [-44] -> [-42]
  BLOCKED_STORM_EG: [-32] -> [-31]
  PAWNLESS_FLANK: [-12, -57] -> [-12, -56]
Summary: 7 numeric change(s), 106 identical (skipped), 113 names in tuned dump.
Updated 7 constant(s) in chess_engine/classical/constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Wrote tuner/constants_snapshot.py from chess_engine/classical/constants.py; removed 0 stale classical Numba cache file(s)
[block5b_shelter_storm_king_pst] Accepted 7 parameter group(s) into chess_engine/classical/constants.py. Backup saved to tuner/runs/classical_with_pgn_sf20/

['PST_MG',
 'PST_EG',
 'SHELTER_STRENGTH',
 'UNBLOCKED_STORM',
 'BLOCKED_STORM',
 'BLOCKED_STORM_EG',
 'PAWNLESS_FLANK']

In [38]:
# [Run] Block 5c — 王安全輸出尺度與殘局安全縮放 (auto_scale=3.3~512；STATIC_FEATURES_ONLY=True 自動 SKIPPED)
res_5c = session.run_block(
    "block5c_king_safety_output",
    tune=[
        "KING_DANGER_OUT_MG_NUM",
        "KING_DANGER_OUT_EG_NUM",
        "EG_SAFETY_SCALE",
    ],
    iters=2500,
    c=2.0,
    c_floor=1.25,
    alpha=22000.0,
    A=150.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=(0 if getattr(session, "random_seed", False) else 503),
)

=== Skipping Block [block5c_king_safety_output] (static_features_only=True; non-static params frozen: KING_DANGER_OUT_MG_NUM, KING_DANGER_OUT_EG_NUM, EG_SAFETY_SCALE) ===


In [39]:
# [Accept] Block 5c
session.accept_if_pass("block5c_king_safety_output")


[SKIP] [block5c_king_safety_output] Gate 判定為 'SKIPPED (static-only)' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

---
## Block 6：單棋種 子力價值 + PST 聯合精調（`N` → `B` → `R` → `Q` → `P` → `6f Material`）
- `PST_MG` / `PST_EG` 每列固定 `theta0` 平均值（量化整數和守恆），因此可直接在單棋種階段將對應的 `MG_MATERIAL_VALUES` / `EG_MATERIAL_VALUES` 與該棋種 128 格 `PST` 同步共調（共 130 維；`6e_pawn` 僅解鎖 `EG Pawn` ±10% 共 97 維，`MG Pawn = 100` 永遠固定為全引擎錨點），避免前面區塊改動 `Mobility` / `Outposts` / `Passed` 後讓過時子力價值拖累單子形狀收斂。
- 因單一棋子落在單格的出現率僅 `< 1% ~ 1.5%` 且 Rademacher 投影雜訊較大，`6a–6d` 採 `batch_size=131072, A=250, iters=4500~6000`（`6c_rook=6000, 6d_queen=5500`）與 `alpha=135000~155000`；`6e_pawn` 採 `iters=4000, batch_size=131072, alpha=25000, A=150`。
- **Block 6f (`block6f_material`)**：僅針對 `MG_MATERIAL_VALUES` + `EG_MATERIAL_VALUES` 的純 9 維（`MG N/B/R/Q` + `EG P/N/B/R/Q`）快速獨立校準（`iters=2500, c=2.0, alpha=40000, A=150`），確保即使 `6b`/`6e` 因格位在實戰 `val_pgn` 受阻，所有子力基準比值仍能在無格位雜訊下獨立收斂。

In [40]:
# [Run] Block 6a_knight — 騎士 (Knight) 子力 + PST 聯合調優 (130 params, batch=131072, alpha=135000, A=250)
res_6a_knight = session.run_block(
    "block6a_knight",
    tune=["MG_MATERIAL_VALUES", "EG_MATERIAL_VALUES", "PST_MG", "PST_EG"],
    pst_pieces="N",
    iters=4500,
    c=2.5,
    alpha=135000.0,
    A=250.0,
    batch_size=131072,
    report_every=50,
    patience=35,
    seed=(0 if getattr(session, "random_seed", False) else 601),
)

  [SPSA Seed] Using dynamic time seed: 1684698506
=== Running Block [block6a_knight] -> tuner/runs/classical_with_pgn_sf20/block6a_knight.py (K=1.156, labels=SF_mix(lambda=0.20)) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1213 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.5004, std=0.3466
  Continuous/mixed labels detected (1001 unique values).
PST-piece mode active; also: ['MG_MATERIAL_VALUES', 'EG_MATERIAL_VALUES']
PST piece filter: [N] rows active on PST_MG/EG
MG_MATERIAL_VALUES: keep Pawn/King frozen (indices 0,5)
EG_MATERIAL_VALUES: keep King frozen (index 5)
Active parameters: 130 / 1276
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.156): 0.0648741
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0392163  n=74,346
    phase=1..31    

In [41]:
# [Accept] Block 6a_knight
session.accept_if_pass("block6a_knight")


[PASS] [block6a_knight] Gate 判定為 PASS -> 正在套用變更至 chess_engine/classical/constants.py 並建立備份...
  PST_MG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
  PST_EG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
Summary: 2 numeric change(s), 111 identical (skipped), 113 names in tuned dump.
Updated 2 constant(s) in chess_engine/classical/constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Wrote tuner/constants_snapshot.py from chess_engine/classical/constants.py; removed 0 stale classical Numba cache file(s)
[block6a_knight] Accepted 2 parameter group(s) into chess_engine/classical/constants.py. Backup saved to tuner/runs/classical_with_pgn_sf20/backups/20261116_130814_classical_before_block6a_knight_constants.py.
[Drive Backup] (block6a_knight_accept) Synced -> /content/drive/MyDrive/chess_numba_data/texel_runs_static_sf20/classical_with_pgn_sf20


['PST_MG', 'PST_EG']

In [42]:
# [Run] Block 6b_bishop — 主教 (Bishop) 子力 + PST 聯合調優 (130 params, batch=131072, alpha=135000, A=250)
res_6b_bishop = session.run_block(
    "block6b_bishop",
    tune=["MG_MATERIAL_VALUES", "EG_MATERIAL_VALUES", "PST_MG", "PST_EG"],
    pst_pieces="B",
    iters=4500,
    c=2.5,
    alpha=135000.0,
    A=250.0,
    batch_size=131072,
    report_every=50,
    patience=35,
    seed=(0 if getattr(session, "random_seed", False) else 602),
)

  [SPSA Seed] Using dynamic time seed: 1685655024
=== Running Block [block6b_bishop] -> tuner/runs/classical_with_pgn_sf20/block6b_bishop.py (K=1.156, labels=SF_mix(lambda=0.20)) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1213 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.5004, std=0.3466
  Continuous/mixed labels detected (1001 unique values).
PST-piece mode active; also: ['MG_MATERIAL_VALUES', 'EG_MATERIAL_VALUES']
PST piece filter: [B] rows active on PST_MG/EG
MG_MATERIAL_VALUES: keep Pawn/King frozen (indices 0,5)
EG_MATERIAL_VALUES: keep King frozen (index 5)
Active parameters: 130 / 1276
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.156): 0.0648678
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0392140  n=74,346
    phase=1..31    

In [43]:
# [Accept] Block 6b_bishop
session.accept_if_pass("block6b_bishop")


[SKIP] [block6b_bishop] Gate 判定為 'REJECT' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

In [44]:
# [Run] Block 6c_rook — 城堡 (Rook) 子力 + PST 聯合調優 (130 params, iters=6000, batch=131072, alpha=140000, A=250)
res_6c_rook = session.run_block(
    "block6c_rook",
    tune=["MG_MATERIAL_VALUES", "EG_MATERIAL_VALUES", "PST_MG", "PST_EG"],
    pst_pieces="R",
    iters=6000,
    c=2.5,
    alpha=140000.0,
    A=250.0,
    batch_size=131072,
    report_every=50,
    patience=45,
    seed=(0 if getattr(session, "random_seed", False) else 603),
)

  [SPSA Seed] Using dynamic time seed: 1686593438
=== Running Block [block6c_rook] -> tuner/runs/classical_with_pgn_sf20/block6c_rook.py (K=1.156, labels=SF_mix(lambda=0.20)) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1213 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.5004, std=0.3466
  Continuous/mixed labels detected (1001 unique values).
PST-piece mode active; also: ['MG_MATERIAL_VALUES', 'EG_MATERIAL_VALUES']
PST piece filter: [R] rows active on PST_MG/EG
MG_MATERIAL_VALUES: keep Pawn/King frozen (indices 0,5)
EG_MATERIAL_VALUES: keep King frozen (index 5)
Active parameters: 130 / 1276
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.156): 0.0648678
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0392140  n=74,346
    phase=1..31        

In [45]:
# [Accept] Block 6c_rook
session.accept_if_pass("block6c_rook")


[PASS] [block6c_rook] Gate 判定為 PASS -> 正在套用變更至 chess_engine/classical/constants.py 並建立備份...
  MG_MATERIAL_VALUES: [100, 353, 361, 570, 1175, 0] -> [100, 353, 361, 567, 1175, 0]
  EG_MATERIAL_VALUES: [117, 418, 439, 698, 1299, 0] -> [117, 418, 439, 700, 1299, 0]
  PST_MG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
  PST_EG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
Summary: 4 numeric change(s), 109 identical (skipped), 113 names in tuned dump.
Updated 4 constant(s) in chess_engine/classical/constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Wrote tuner/constants_snapshot.py from chess_engine/classical/constants.py; removed 0 stale classical Numba cache file(s)
[block6c_rook] Accepted 4 parameter group(s) into chess_engine/classical/constants.py. Backup saved to tuner/runs/classical_with_pgn_sf20/backups/20261116_134543_classical_before_block6c_rook_constants.py.
[Drive Backup] (block6c_rook_accept) Synced -> /content/drive/M

['MG_MATERIAL_VALUES', 'EG_MATERIAL_VALUES', 'PST_MG', 'PST_EG']

In [46]:
# [Run] Block 6d_queen — 皇后 (Queen) 子力 + PST 聯合調優 (130 params, 單格稀疏度最高，iters=5500, batch=131072, alpha=155000, A=250)
res_6d_queen = session.run_block(
    "block6d_queen",
    tune=["MG_MATERIAL_VALUES", "EG_MATERIAL_VALUES", "PST_MG", "PST_EG"],
    pst_pieces="Q",
    iters=5500,
    c=2.5,
    alpha=155000.0,
    A=250.0,
    batch_size=131072,
    report_every=50,
    patience=40,
    seed=(0 if getattr(session, "random_seed", False) else 604),
)

  [SPSA Seed] Using dynamic time seed: 1687905379
=== Running Block [block6d_queen] -> tuner/runs/classical_with_pgn_sf20/block6d_queen.py (K=1.156, labels=SF_mix(lambda=0.20)) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1213 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.5004, std=0.3466
  Continuous/mixed labels detected (1001 unique values).
PST-piece mode active; also: ['MG_MATERIAL_VALUES', 'EG_MATERIAL_VALUES']
PST piece filter: [Q] rows active on PST_MG/EG
MG_MATERIAL_VALUES: keep Pawn/King frozen (indices 0,5)
EG_MATERIAL_VALUES: keep King frozen (index 5)
Active parameters: 130 / 1276
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.156): 0.0648575
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0392000  n=74,346
    phase=1..31      

In [47]:
# [Accept] Block 6d_queen
session.accept_if_pass("block6d_queen")


[PASS] [block6d_queen] Gate 判定為 PASS -> 正在套用變更至 chess_engine/classical/constants.py 並建立備份...
  MG_MATERIAL_VALUES: [100, 353, 361, 567, 1175, 0] -> [100, 353, 361, 567, 1178, 0]
  EG_MATERIAL_VALUES: [117, 418, 439, 700, 1299, 0] -> [117, 418, 439, 700, 1311, 0]
  PST_MG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
  PST_EG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
Summary: 4 numeric change(s), 109 identical (skipped), 113 names in tuned dump.
Updated 4 constant(s) in chess_engine/classical/constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Wrote tuner/constants_snapshot.py from chess_engine/classical/constants.py; removed 0 stale classical Numba cache file(s)
[block6d_queen] Accepted 4 parameter group(s) into chess_engine/classical/constants.py. Backup saved to tuner/runs/classical_with_pgn_sf20/backups/20261116_140339_classical_before_block6d_queen_constants.py.
[Drive Backup] (block6d_queen_accept) Synced -> /content/dri

['MG_MATERIAL_VALUES', 'EG_MATERIAL_VALUES', 'PST_MG', 'PST_EG']

In [48]:
# [Run] Block 6e_pawn — 士兵 EG 子力 + PST 調優 (97 params: MG Pawn=100 鎖定, EG Pawn±10% + Rank2-7, iters=4000, batch=131072, alpha=25000, A=150)
res_6e_pawn = session.run_block(
    "block6e_pawn",
    tune=["EG_MATERIAL_VALUES", "PST_MG", "PST_EG"],
    pst_pieces="P",
    iters=4000,
    c=2.5,
    alpha=25000.0,
    A=150.0,
    batch_size=131072,
    report_every=50,
    patience=30,
    seed=(0 if getattr(session, "random_seed", False) else 605),
)

  [SPSA Seed] Using dynamic time seed: 1688979726
=== Running Block [block6e_pawn] -> tuner/runs/classical_with_pgn_sf20/block6e_pawn.py (K=1.157, labels=SF_mix(lambda=0.20)) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1213 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.5004, std=0.3466
  Continuous/mixed labels detected (1001 unique values).
PST-piece mode active; also: ['EG_MATERIAL_VALUES']
PST piece filter: [P] rows active on PST_MG/EG
EG_MATERIAL_VALUES: keep King frozen (index 5)
Active parameters: 97 / 1276
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.157): 0.0648334
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0391955  n=74,346
    phase=1..31           0.0542403  n=38,267
    phase=32..63          0.0680446  n=33,184
    phase=

In [49]:
# [Accept] Block 6e_pawn
session.accept_if_pass("block6e_pawn")


[SKIP] [block6e_pawn] Gate 判定為 'REJECT' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

In [50]:
# [Run] Block 6f_material — 純 9 維全子力價值校準 (MG N/B/R/Q + EG P/N/B/R/Q，無 PST 格位雜訊；iters=2500, batch=131072, alpha=40000, A=150)
res_6f_material = session.run_block(
    "block6f_material",
    tune=["MG_MATERIAL_VALUES", "EG_MATERIAL_VALUES"],
    iters=2500,
    c=2.0,
    alpha=40000.0,
    A=150.0,
    batch_size=131072,
    report_every=50,
    patience=30,
    seed=(0 if getattr(session, "random_seed", False) else 606),
)

  [SPSA Seed] Using dynamic time seed: 1689724442
=== Running Block [block6f_material] -> tuner/runs/classical_with_pgn_sf20/block6f_material.py (K=1.157, labels=SF_mix(lambda=0.20)) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1213 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.5004, std=0.3466
  Continuous/mixed labels detected (1001 unique values).
Freezing all parameters except: ['MG_MATERIAL_VALUES', 'EG_MATERIAL_VALUES']
MG_MATERIAL_VALUES: keep Pawn/King frozen (indices 0,5)
EG_MATERIAL_VALUES: keep King frozen (index 5)
Active parameters: 9 / 1276
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.157): 0.0648334
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0391955  n=74,346
    phase=1..31           0.0542403  n=38,267
    phase=32..6

In [51]:
# [Accept] Block 6f_material
session.accept_if_pass("block6f_material")


[SKIP] [block6f_material] Gate 判定為 'REJECT' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

In [52]:
# session.rollback()

---
## Block 7：材質不平衡、主動權與空間/先手微調（Imbalance, Initiative, Space & Tempo）
- **Block 7a_div_grid**：材質不平衡總除數（`IMBALANCE_DIVISOR=18:22`，Clean R2=20；`STATIC_FEATURES_ONLY=True` 下自動 `SKIPPED`）。
- **Block 7a**：二次材質不平衡矩陣（`STATIC_FEATURES_ONLY=True` 下自動 `SKIPPED`）。
- **Block 7b**：Initiative 複雜度權重與偏置（靜態維持調優；未使用的 `INITIATIVE_PAWN_ENDGAME_WEIGHT_*` 已自動凍結）。
- **Block 7c_grid**：`TEMPO_BONUS` 與空間除數/門檻窮舉掃描（Clean R2 `TEMPO_BONUS=19`, `SPACE_THRESHOLD=6000`；`STATIC_FEATURES_ONLY=True` 下自動 `SKIPPED`）。

In [53]:
# [Run] Block 7a_div_grid — 材質不平衡除數離散掃描 (Clean R2 目前 IMBALANCE_DIVISOR=20；STATIC_FEATURES_ONLY=True 自動 SKIPPED)
res_7a_div = session.scan_grid(
    "block7a_imbalance_div_grid",
    grids=[
        "IMBALANCE_DIVISOR=18:22",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=10,
)

=== Skipping Grid [block7a_imbalance_div_grid] (static_features_only=True; non-static scalars frozen: IMBALANCE_DIVISOR) ===


In [54]:
# [Accept] Block 7a_div_grid
session.accept_if_pass("block7a_imbalance_div_grid")


[SKIP] [block7a_imbalance_div_grid] Gate 判定為 'SKIPPED (static-only)' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

In [55]:
# [Run] Block 7a — 二次材質不平衡矩陣 (Imbalance Polynomial；STATIC_FEATURES_ONLY=True 自動 SKIPPED)
res_7a = session.run_block(
    "block7a_imbalance",
    tune=[
        "IMBALANCE_QUADRATIC_OURS",
        "IMBALANCE_QUADRATIC_THEIRS",
        "IMBALANCE_SCALE_MG",
        "IMBALANCE_SCALE_EG",
    ],
    iters=3000,
    c=2.0,
    alpha=30000.0,
    A=150.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=(0 if getattr(session, "random_seed", False) else 701),
)

=== Skipping Block [block7a_imbalance] (static_features_only=True; non-static params frozen: IMBALANCE_QUADRATIC_OURS, IMBALANCE_QUADRATIC_THEIRS, IMBALANCE_SCALE_MG, IMBALANCE_SCALE_EG) ===


In [56]:
# [Accept] Block 7a
session.accept_if_pass("block7a_imbalance")


[SKIP] [block7a_imbalance] Gate 判定為 'SKIPPED (static-only)' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

In [57]:
# [Run] Block 7b — 主動權複雜度係數 (alpha=12000, A=200, batch=131072 降低殘局 phase=0 雜訊)
res_7b = session.run_block(
    "block7b_initiative",
    tune=[
        "INITIATIVE_PASSED_WEIGHT_MG",
        "INITIATIVE_PAWN_WEIGHT_MG",
        "INITIATIVE_OUTFLANKING_WEIGHT_MG",
        "INITIATIVE_INFILTRATION_WEIGHT_MG",
        "INITIATIVE_BOTH_FLANKS_WEIGHT_MG",
        "INITIATIVE_ALMOST_UNWIN_WEIGHT_MG",
        "INITIATIVE_OFFSET_MG",
        "INITIATIVE_MG_OFFSET",
        "INITIATIVE_PASSED_WEIGHT_EG",
        "INITIATIVE_PAWN_WEIGHT_EG",
        "INITIATIVE_OUTFLANKING_WEIGHT_EG",
        "INITIATIVE_INFILTRATION_WEIGHT_EG",
        "INITIATIVE_BOTH_FLANKS_WEIGHT_EG",
        "INITIATIVE_ALMOST_UNWIN_WEIGHT_EG",
        "INITIATIVE_OFFSET_EG",
    ],
    iters=2500,
    c=1.0,
    c_floor=1.0,
    alpha=12000.0,
    A=200.0,
    batch_size=131072,
    report_every=50,
    patience=30,
    seed=(0 if getattr(session, "random_seed", False) else 702),
)

  [SPSA Seed] Using dynamic time seed: 1690190922
=== Running Block [block7b_initiative] -> tuner/runs/classical_with_pgn_sf20/block7b_initiative.py (K=1.157, labels=SF_mix(lambda=0.20)) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1213 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.5004, std=0.3466
  Continuous/mixed labels detected (1001 unique values).
Freezing all parameters except: ['INITIATIVE_PASSED_WEIGHT_MG', 'INITIATIVE_PAWN_WEIGHT_MG', 'INITIATIVE_OUTFLANKING_WEIGHT_MG', 'INITIATIVE_INFILTRATION_WEIGHT_MG', 'INITIATIVE_BOTH_FLANKS_WEIGHT_MG', 'INITIATIVE_ALMOST_UNWIN_WEIGHT_MG', 'INITIATIVE_OFFSET_MG', 'INITIATIVE_MG_OFFSET', 'INITIATIVE_PASSED_WEIGHT_EG', 'INITIATIVE_PAWN_WEIGHT_EG', 'INITIATIVE_OUTFLANKING_WEIGHT_EG', 'INITIATIVE_INFILTRATION_WEIGHT_EG', 'INITIATIVE_BOTH_FLANKS_WEIGHT_EG', 'INITIATIVE_ALMOST_UNWIN_WEIGHT_EG', 'INITIATIVE_OFFSET_EG']
Active parameters: 15 / 1276
Coordinate auto-

In [58]:
# [Accept] Block 7b
session.accept_if_pass("block7b_initiative")


[SKIP] [block7b_initiative] Gate 判定為 'REJECT' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

In [59]:
# [Run] Block 7c_grid — Tempo、Space 門檻與除數離散掃描 (Clean R2 置中於 SPACE_THRESHOLD=6000, TEMPO_BONUS=19；STATIC_FEATURES_ONLY=True 自動 SKIPPED)
res_7c = session.scan_grid(
    "block7c_space_tempo_grid",
    grids=[
        "SPACE_BONUS_DIVISOR=14:16",
        "SPACE_SCALE_DIVISOR=5:7",
        "SPACE_THRESHOLD=5800:6200:100",
        "TEMPO_BONUS=18:20",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=10,
)

=== Skipping Grid [block7c_space_tempo_grid] (static_features_only=True; non-static scalars frozen: SPACE_BONUS_DIVISOR, SPACE_SCALE_DIVISOR, SPACE_THRESHOLD, TEMPO_BONUS) ===


In [60]:
# [Accept] Block 7c_grid — 可用 only=[...] 只挑確認有效的常數套用
session.accept_if_pass("block7c_space_tempo_grid")


[SKIP] [block7c_space_tempo_grid] Gate 判定為 'SKIPPED (static-only)' (非 PASS) -> 自動跳過，保持既有 chess_engine/classical/constants.py 參數不變。


[]

---
## Block 8：全子力聯合收尾 Polish（`NBRQ` Material + PST Joint Polish）
在前面已調整過單棋種 Material + PST、純 9 維 Material 校準與殘餘靜態區塊後，以 `batch_size=131072, alpha=65000, A=250, c=2.0, iters=6000` 同時微調 `N/B/R/Q` 的材質與 PST（共 520 自由度），消除跨棋種之間的殘餘交互作用。

In [61]:
# [Run] Block 8 — NBRQ Material + PST 聯合 Polish (520 參數，iters=6000, batch=131072, alpha=65000, A=250)
res_8 = session.run_block(
    "block8_nbrq_joint_polish",
    tune=["MG_MATERIAL_VALUES", "EG_MATERIAL_VALUES", "PST_MG", "PST_EG"],
    pst_pieces="NBRQ",
    iters=6000,
    c=2.0,
    alpha=65000.0,
    A=250.0,
    batch_size=131072,
    report_every=50,
    patience=40,
    seed=(0 if getattr(session, "random_seed", False) else 801),
)

  [SPSA Seed] Using dynamic time seed: 1690497850
=== Running Block [block8_nbrq_joint_polish] -> tuner/runs/classical_with_pgn_sf20/block8_nbrq_joint_polish.py (K=1.157, labels=SF_mix(lambda=0.20)) ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1213 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.5004, std=0.3466
  Continuous/mixed labels detected (1001 unique values).
PST-piece mode active; also: ['MG_MATERIAL_VALUES', 'EG_MATERIAL_VALUES']
PST piece filter: [N,B,R,Q] rows active on PST_MG/EG
MG_MATERIAL_VALUES: keep Pawn/King frozen (indices 0,5)
EG_MATERIAL_VALUES: keep King frozen (index 5)
Active parameters: 520 / 1276
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.157): 0.0648334
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0391955  n=

In [62]:
# [Accept] Block 8
session.accept_if_pass("block8_nbrq_joint_polish")


[PASS] [block8_nbrq_joint_polish] Gate 判定為 PASS -> 正在套用變更至 chess_engine/classical/constants.py 並建立備份...
  MG_MATERIAL_VALUES: [100, 353, 361, 567, 1178, 0] -> [100, 353, 361, 567, 1179, 0]
  EG_MATERIAL_VALUES: [117, 418, 439, 700, 1311, 0] -> [117, 418, 439, 701, 1315, 0]
  PST_MG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
  PST_EG: [0, 0, 0, 0, 0, 0] -> [0, 0, 0, 0, 0, 0]
Summary: 4 numeric change(s), 109 identical (skipped), 113 names in tuned dump.
Updated 4 constant(s) in chess_engine/classical/constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Wrote tuner/constants_snapshot.py from chess_engine/classical/constants.py; removed 0 stale classical Numba cache file(s)
[block8_nbrq_joint_polish] Accepted 4 parameter group(s) into chess_engine/classical/constants.py. Backup saved to tuner/runs/classical_with_pgn_sf20/backups/20261116_144920_classical_before_block8_nbrq_joint_polish_constants.py.
[Drive Backup] (block8_nbrq

['MG_MATERIAL_VALUES', 'EG_MATERIAL_VALUES', 'PST_MG', 'PST_EG']

---
## 9. 總結驗收、回滾工具與分階段對戰流程
完成一輪調參後：
1. 執行下方 **`session.cumulative_report()`** 檢視相較於本次啟動前初始 `theta0` 的累積變動與驗證集總增益。
2. 執行 **Cell `9c`** 依 `session.target_engine` 同步快取並通過一致性與殘局不變量單元測試。
3. **分階段實驗與對戰驗收路線**：
   - **階段一（Clean R2 基準 + 純 WDL 靜態特徵調優 vs 基準 `classical_old`）**：`# 0` 預設 `STATIC_FEATURES_ONLY = True, USE_PGN_VAL = True, USE_ZURICHESS_VAL = True, USE_SF_LABELS = False, TARGET_ENGINE = "classical"`，從 Clean R2（`classical_old/constants.py`）重新僅調優 66 組靜態特徵（非靜態與搜尋耦合區塊自動 `SKIPPED`；若 `.sf19.npz` 覆蓋 Holdout / Zurichess 會自動附帶純 WDL `sf19-quiet (WDL)` 子診斷），完成後於本機與 `classical_old` 對戰驗收：
     `python tools/tournament.py -c 4 --games 500 --nodes 300000 --name1 New_StaticOnly --name2 CleanR2_Base`
   - **階段二（對照實驗與 Match-SPSA 擴展）**：若靜態特徵調優對戰明確勝出並經同意更新基準，後續再行評估驗證集條件或動態/搜尋耦合參數（應以實戰對局 / Match-SPSA 驗證為主）。
   - **階段三（SF19 搜尋 + WDL 混合標籤實驗，選用）**：先於終端機執行 `python -m tuner.data_pipeline.annotate_sf_labels --nodes 4000 --skip-holdout`，再於 `# 0` 啟用 `USE_SF_LABELS = True, SF_LAMBDA = 0.15`（或調整靜態特徵對照），最終一律以實戰對局 Elo 為準。

In [63]:
# 9a. 累積總成績單（目前 constants.py 對比本次 Session 啟動前的初始 theta0）
cum_stats = session.cumulative_report()

=== Cumulative Progress vs Initial Session Baseline (theta0) [target=classical -> chess_engine/classical/constants.py | use_pgn_val=True] ===
Effective integer changes:
  MG_MATERIAL_VALUES: changed=2/6, L1=7
  EG_MATERIAL_VALUES: changed=2/6, L1=19
  PST_MG: changed=112/384, L1=150
  PST_EG: changed=143/384, L1=176
  KNIGHT_MOBILITY_BONUS: changed=2/18, L1=2
  BISHOP_MOBILITY_BONUS: changed=7/28, L1=7
  ROOK_MOBILITY_BONUS: changed=18/30, L1=21
  QUEEN_MOBILITY_BONUS: changed=24/56, L1=33
  CONNECTED_SUPPORT_WEIGHT: 14 -> 15
  CONNECTED_SUPPORT_WEIGHT_EG: 5 -> 4
  SHELTER_STRENGTH: changed=9/32, L1=10
  UNBLOCKED_STORM: changed=10/32, L1=18
  BLOCKED_STORM: -44 -> -42
  BLOCKED_STORM_EG: -32 -> -31
  PAWNLESS_FLANK: changed=1/2, L1=1

[dataset_epd_wdl_20m.npz [cumulative vs start, pure WDL]] Pre-selected indices (2,000,000): n=2,000,000 (K=1.239)
  processed 500,000/2,000,000
  processed 1,000,000/2,000,000
  processed 1,500,000/2,000,000
  processed 2,000,000/2,000,000
Paired candida

In [64]:
# 9b. 【選用工具】回滾上一步 或 手動覆寫單一參數重新驗證
# 若要撤銷最近一次 accept()：
# session.rollback()

# 若要撤銷到特定區塊之前的備份（例如撤銷 block5b 及其之後的 accept）：
# session.rollback("block5b_shelter_storm_king_pst")

# 若要測試某個候選檔加上手動微調單一值：
# session.validate_file("block5a_king_danger", overrides=["KING_DANGER_NO_QUEEN=570"])

In [65]:
# # 9c. 依 target_engine 同步快取並執行評估一致性與殘局不變量單元測試
# import subprocess

# target_eng = getattr(session, "target_engine", "classical")
# if target_eng == "classical":
#     cmds = [
#         [sys.executable, "-m", "tuner.sync_constants_snapshot"],
#         [sys.executable, "-m", "tuner.codegen.generate_tunable_eval"],
#         [
#             sys.executable, "-m", "unittest",
#             "tests.test_texel_parameterization",
#             "tests.test_endgame_conformance",
#             "tests.test_tunable_eval_parity",
#             "tests.test_runtime_eval_weights",
#             "-v",
#         ],
#     ]
# else:
#     parity_snippet = (
#         "from pathlib import Path; "
#         "from chess_engine.classical.fen_parser import parse_fen; "
#         "from chess_engine.classical_old.evaluation import evaluate_position as eval_old; "
#         "from tuner.parameters import ParameterManager; "
#         "from tuner.tunable_eval import evaluate_position_tunable; "
#         "from tuner.sync_constants_snapshot import clear_classical_numba_cache; "
#         "clear_classical_numba_cache(Path('chess_engine/classical_old/__pycache__')); "
#         "theta = ParameterManager('chess_engine/classical_old/constants.py').get_initial_theta(); "
#         "fens = ["
#         "'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1',"
#         "'r3k2r/p1ppqpb1/bn2pnp1/3PN3/1p2P3/2N2Q1p/PPPBBPPP/R3K2R w KQkq - 0 1',"
#         "'8/2p5/3p4/KP5r/1R3p1k/8/4P1P1/8 w - - 0 1'"
#         "]; "
#         "diffs = []; "
#         "[diffs.append((f, int(eval_old(*parse_fen(f), lazy=False)[0]), int(evaluate_position_tunable(*parse_fen(f), theta, False)))) for f in fens]; "
#         "bad = [d for d in diffs if d[1] != d[2]]; "
#         "assert not bad, f'classical_old parity mismatch: {bad}'; "
#         "print('classical_old parity OK:', [(d[1], d[2]) for d in diffs])"
#     )
#     cmds = [
#         [sys.executable, "-m", "unittest", "tests.test_texel_parameterization", "-v"],
#         [sys.executable, "-c", parity_snippet],
#     ]

# for cmd in cmds:
#     print("+", " ".join(str(x) for x in cmd[:4]), "..." if len(cmd) > 4 else "", flush=True)
#     with subprocess.Popen(
#         cmd,
#         cwd=str(REPO_ROOT),
#         stdout=subprocess.PIPE,
#         stderr=subprocess.STDOUT,
#         text=True,
#         bufsize=1,
#     ) as proc:
#         assert proc.stdout is not None
#         for line in proc.stdout:
#             print(line, end="")
#         rc = proc.wait()
#     if rc != 0:
#         raise RuntimeError(f"Command failed with exit code {rc}: {cmd}")
# print(f"✅ [{target_eng}] 一致性與參數化測試通過！可進行對局閘門驗證：")
# print("   python tools/tournament.py -c 4 --games 500 --nodes 300000 --name1 New_StaticOnly --name2 CleanR2_Base")

In [66]:
# 9d. 【Colab / Google Drive】完整同步與打包下載最新調參結果
from datetime import datetime
import zipfile

target_eng = getattr(session, "target_engine", "classical")
target_const = Path(getattr(session, "target_constants_path", REPO_ROOT / "chess_engine" / target_eng / "constants.py"))
runs_dir = Path(getattr(session, "runs_dir", REPO_ROOT / "tuner" / "runs"))

session.sync_to_drive(tag="final")
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f"texel_tuned_{target_eng}_{runs_dir.name}_{ts}.zip"

out_root = globals().get("DRIVE_BACKUP_DIR") or runs_dir
out_root = Path(out_root)
out_root.mkdir(parents=True, exist_ok=True)
zip_path = out_root / zip_name

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    if target_const.is_file():
        zf.write(target_const, arcname=f"chess_engine/{target_eng}/constants.py")
    for py_file in sorted(runs_dir.glob("*.py")):
        zf.write(py_file, arcname=f"tuner/runs/{runs_dir.name}/{py_file.name}")
    snap = REPO_ROOT / "tuner" / "constants_snapshot.py"
    if target_eng == "classical" and snap.is_file():
        zf.write(snap, arcname="tuner/constants_snapshot.py")

print(f"📦 已匯出調參成果壓縮包：{zip_path}")
if target_const.is_file():
    print(f"📄 目標引擎常數檔：{target_const}")
if globals().get("DRIVE_BACKUP_DIR"):
    print(f"☁️ Google Drive 已保留未壓縮常數檔：{Path(DRIVE_BACKUP_DIR) / (target_eng + '_constants.py')}")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass

[Drive Backup] (final) Synced -> /content/drive/MyDrive/chess_numba_data/texel_runs_static_sf20/classical_with_pgn_sf20
📦 已匯出調參成果壓縮包：/content/drive/MyDrive/chess_numba_data/texel_runs_static_sf20/texel_tuned_classical_classical_with_pgn_sf20_20261116_144931.zip
📄 目標引擎常數檔：/content/chess_numba/chess_engine/classical/constants.py
☁️ Google Drive 已保留未壓縮常數檔：/content/drive/MyDrive/chess_numba_data/texel_runs_static_sf20/classical_constants.py


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [67]:
# =====================================================================
# 嚴格核對新舊引擎代碼差異（驗證 Candidate vs Baseline）
# =====================================================================
import os
import sys
from pathlib import Path

if not (Path("chess_engine").is_dir() and Path("tools").is_dir()):
    if (Path.cwd().parent / "chess_engine").is_dir():
        os.chdir(Path.cwd().parent)
    elif Path("/content/chess_numba").is_dir():
        os.chdir("/content/chess_numba")

# 1) 清理 Python 模組記憶體快取
for m in list(sys.modules.keys()):
    if 'chess_engine' in m or 'tune_search' in m:
        del sys.modules[m]
!python -m tuner.sync_constants_snapshot
# 2) 顯示新舊版本差異（確認 classical vs classical_old 確實包含本次修改）
print("=" * 60)
print("🔍 新舊引擎邏輯代碼差異比對 (classical vs classical_old)：")
print("=" * 60)
!python tools/sync_classical_old.py --diff

# 3) 快速合約測試 (確保 JIT 與核心介面運作正常)
print("\n" + "=" * 60)
print("🔥 執行單元合約測試：")
print("=" * 60)
!python -m unittest tests/test_search_core_contracts.py

Unchanged tuner/constants_snapshot.py
🔍 新舊引擎邏輯代碼差異比對 (classical vs classical_old)：

[!] Differences found in constants.py:
--- classical_old/constants.py
+++ classical/constants.py
@@ -40,8 +40,8 @@
 #   SF types.h → N 367/401, B 387/430, R 599/648, Q 1192/1259.
 # Passed / path / prox use the same *100/213 so pass/N matches SF relative strength.
 # Index: PAWN, KNIGHT, BISHOP, ROOK, QUEEN, KING
-MG_MATERIAL_VALUES = np.array([100, 353, 361, 570, 1175, 0], dtype=np.int32)
-EG_MATERIAL_VALUES = np.array([117, 418, 439, 698, 1299, 0], dtype=np.int32)
+MG_MATERIAL_VALUES = np.array([100, 353, 361, 567, 1179, 0], dtype=np.int32)
+EG_MATERIAL_VALUES = np.array([117, 418, 439, 701, 1315, 0], dtype=np.int32)
 # =============================================================================
 # --- Game Phase Calculation (SF11 npm taper) / 遊戲階段計算 ---
 # =============================================================================
@@ -77,19 +77,19 @@
 
 PST_MG = np.array([
     [0, 0, 0, 0, 0, 0, 

In [ ]:
# =====================================================================
# 步驟 4: 設定對戰參數並啟動錦標賽
# =====================================================================
import multiprocessing
import os
import time
from pathlib import Path

# 自動偵測可用的 CPU 核心數 (留 1 核給主行程排程)
cpu_count = multiprocessing.cpu_count()
auto_concurrency = max(1, cpu_count - 1)

# --- 參數設定 ---
GAMES = 6000                 # 對局總數 (必須為偶數；400 局可獲穩健 Elo 95% CI)
NODES = 400000              # 每步節點預算 (300k 長時控；快速測試可改為 50000)
TT_MB = 64                  # 每個 Worker 的轉置表大小 (MB)
CONCURRENCY = auto_concurrency  # 平行對局數 (例如 7 或 11)
SEED = time.time_ns()

print("=" * 60)
print(f"♟️ 錦標賽參數配置：")
print(f"  總局數: {GAMES} 局 ({GAMES // 2} 對成對換先開局)" 
      f"  時控: {NODES:,} nodes/move" 
      f"  並行 Worker: {CONCURRENCY}" 
      f"  轉置表: {TT_MB} MB / context")
print("=" * 60)

if not (Path("chess_engine").is_dir() and Path("tools").is_dir()):
    if (Path.cwd().parent / "chess_engine").is_dir():
        os.chdir(Path.cwd().parent)
    elif Path("/content/chess_numba").is_dir():
        os.chdir("/content/chess_numba")
!python tools/tournament.py --games {GAMES} --nodes {NODES} --concurrency {CONCURRENCY} --tt-mb {TT_MB} --seed {SEED}

♟️ 錦標賽參數配置：
  總局數: 6000 局 (3000 對成對換先開局)  時控: 400,000 nodes/move  並行 Worker: 7  轉置表: 64 MB / context
Backend: inprocess; workers=7; TT=64 MB/context; openings=34700; seed=1794841020231339599
Adjudication: win >= 500 cp (10 plies), draw <= 8 cp (12 plies after move 40)
JIT plan: classical once + classical_old once; worker contexts share compiled code.
Warm-up: Kiwipete depth 14 for each Python/Numba backend, then one complete startpos game at depth 1 (all results discarded).
[pool] allocate worker 1/7
[warmup] New: Kiwipete depth=14...
info depth 1 score cp 76 nodes 447 nps 866 hashfull 0.0% time 515 pv e2a6
info depth 2 score cp 76 nodes 865 nps 1102 hashfull 0.0% time 784 pv e2a6 b4c3
info depth 3 score cp -9 nodes 2777 nps 3531 hashfull 0.0% time 786 pv e2a6 b4c3 d2c3
info depth 4 score cp -9 nodes 3112 nps 3955 hashfull 0.0% time 786 pv e2a6 b4c3 d2c3 h3g2
info depth 5 score cp -106 nodes 9106 nps 11482 hashfull 0.0% time 793 pv e2a6 b4c3 d2c3 e6d5 e4d5
info depth 6 score cp -106 no

In [ ]:
# =====================================================================
# 產出統計報告與搜尋剪枝指標診斷
# =====================================================================
import os
from pathlib import Path

if not (Path("chess_engine").is_dir() and Path("tools").is_dir()):
    if (Path.cwd().parent / "chess_engine").is_dir():
        os.chdir(Path.cwd().parent)
    elif Path("/content/chess_numba").is_dir():
        os.chdir("/content/chess_numba")

!python tournament_analysis/統計數據.py
!python tournament_analysis/parse_search_stats.py


In [ ]:
# =====================================================================
# 視覺化關鍵圖表展示
# =====================================================================
import os
from pathlib import Path
from IPython.display import Image, display

if not (Path("chess_engine").is_dir() and Path("tools").is_dir()):
    if (Path.cwd().parent / "chess_engine").is_dir():
        os.chdir(Path.cwd().parent)
    elif Path("/content/chess_numba").is_dir():
        os.chdir("/content/chess_numba")

charts = [
    ("累積分數曲線 (Cumulative Score)", "tournament_analysis/cumulative_score.png"),
    ("累積勝場曲線 (Cumulative Win)", "tournament_analysis/cumulative_wins.png"),
    ("Elo 置信區間收斂圖 (Elo CI)", "tournament_analysis/elo_ci.png"),
    ("每手搜尋深度 (Depth per Move)", "tournament_analysis/depth_per_move.png"),
    ("同深度節點比例分佈 (Same-Depth Node Ratio)", "tournament_analysis/same_depth_node_ratio.png"),
    ("平均節點數 vs 深度散佈 (Avg Nodes vs Depth)", "tournament_analysis/avg_nodes_vs_depth.png"),
    ("黑白勝負分佈 (WDL by Color)", "tournament_analysis/wdl_by_color.png"),
]

for title, path in charts:
    if os.path.exists(path):
        print(f"\n📈 {title}：")
        display(Image(filename=path))

In [ ]:
from google.colab import runtime

# 執行這行會直接中斷並釋放目前使用的虛擬機器運算資源
runtime.unassign()
